# 🍎 Phi-4-reasoning Model with AIProjectClient 🍏

Phi-4-reasoning is an open-weight reasoning model from Microsoft (14B parameters) that is fine-tuned to think step by step, delivering strong results on math, coding, and STEM tasks while staying small enough to run cost-effectively.

In this notebook, you'll see how to:
- Initialize an `AIProjectClient` for your Microsoft Foundry environment.
- Chat with the Phi-4-reasoning model using the project's OpenAI client (`get_openai_client()`) via the **Responses API**.
- Correctly separate the model's internal reasoning trace from its final answer.
- Show a Health & Fitness example, featuring disclaimers and wellness Q&A.

Enjoy strong step-by-step responses from a small, cost-effective model. 🏋️

**Disclaimer:** This is not medical advice. Please consult professionals.

## Why Phi-4-reasoning?
- **Small but Capable:** 14B parameters deliver quality competitive with much larger models, at a fraction of the cost.
- **Reasoning-tuned:** Trained to work through problems step by step, which helps on math, planning, and logic.
- **Generous Context Window:** Enough room for multi-turn conversations.
- **Open Weight:** Available in the Foundry catalog for flexible deployment.

## 1. Setup

We'll import the necessary libraries:
- `azure-ai-projects`: For the endpoint-based `AIProjectClient`.
- `openai` (via `project.get_openai_client()`): For Responses API calls against the deployed model.
- `azure-identity`: For `DefaultAzureCredential`.

Ensure you have a `.env` file with:
```
PROJECT_ENDPOINT=<your-project-endpoint>
MICROSOFT_MODEL=Phi-4-reasoning
```

> **Note:** It's recommended to complete the `3-basic-rag.ipynb` notebook before this one, as it covers important concepts that will be helpful here.

In [ ]:
import os
import re
from dotenv import load_dotenv
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Phi-4-reasoning deployment name (from Build > Models in the Foundry portal)
phi4_deployment = os.getenv("MICROSOFT_MODEL", "Phi-4-reasoning")

try:
    # Endpoint-based client + its OpenAI client (used for Responses API calls)
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ AIProjectClient + OpenAI client created successfully!")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)

## 2. A Note on Reasoning Traces 🧠

Phi-4-reasoning "thinks out loud" before answering. Left unhandled, that internal reasoning trace can leak straight into the text you show a user — full of lines like *"We are asked..."*, *"The instructions say..."*, *"But wait..."* — which looks confusing or broken to anyone reading it.

The trace is typically wrapped in `<think>...</think>` tags, with the real answer following after. We'll write one small helper to split the two apart, and use it everywhere below instead of printing the raw response.

In [ ]:
def split_reasoning_and_answer(text: str):
    """Separate a <think>...</think> reasoning trace from the final answer.

    Returns a dict with 'thinking' and 'answer'. If no <think> block is found,
    'thinking' is None and 'answer' is the full text as-is.
    """
    match = re.search(r"<think>(.*?)</think>(.*)", text, re.DOTALL)
    if match:
        return {"thinking": match.group(1).strip(), "answer": match.group(2).strip()}
    return {"thinking": None, "answer": text.strip()}

## 3. Chat with Phi-4-reasoning 🍏

We'll demonstrate a simple conversation using Phi-4-reasoning in a health & fitness context, via the **Responses API**. We'll define system instructions that clarify the role of the assistant, then ask a user question.

Because this is a reasoning-tuned model, you can optionally ask it to show its work step-by-step — we'll surface that reasoning separately from the clean final answer, using the helper above.

In [ ]:
def chat_with_phi4(user_question: str, step_by_step: bool = False, show_thinking: bool = False):
    """Send a Responses API request to the Phi-4-reasoning model.

    If show_thinking is True, returns a dict with both 'thinking' and 'answer'.
    Otherwise, returns just the clean final answer text.
    """
    system_prompt = (
        "You are a Phi-4-reasoning AI assistant, focusing on health and fitness.\n"
        "Remind users that you are not a medical professional, but can provide general info.\n"
    )

    if step_by_step:
        system_prompt += "Please show your step-by-step reasoning in your answer.\n"

    response = openai_client.responses.create(
        model=phi4_deployment,
        instructions=system_prompt,
        input=user_question,
        temperature=0.8,  # a bit creative
        top_p=0.9,
        max_output_tokens=400,
    )

    parsed = split_reasoning_and_answer(response.output_text)
    return parsed if show_thinking else parsed["answer"]

# Example usage:
question = "I'm training for a 5K. Any tips on a weekly workout schedule?"
answer = chat_with_phi4(question, step_by_step=True, show_thinking=True)
print("🗣️ User:", question)
if answer["thinking"]:
    print("\n🧠 Reasoning:", answer["thinking"])
print("\n🤖 Phi-4-reasoning:", answer["answer"])

## 4. RAG-like Example (Stub) 📚

Phi-4-reasoning also works well in retrieval augmented generation scenarios, where you provide external context and let the model answer over it. Below is a **simplified stub** showing how you'd pass retrieved text as context directly in a Responses API call.

> **Note:** This is a simplified teaching example, not the recommended production RAG pattern. For a real, grounded RAG setup with automatic retrieval and citations, see `3-basic-rag.ipynb`, which uses a Foundry Agent with the Azure AI Search tool. Here, we're manually pasting in a single hardcoded snippet just to show Phi-4-reasoning working over supplied context.

In [ ]:
def chat_with_phi4_rag(user_question: str, retrieved_doc: str, show_thinking: bool = False):
    """Illustrate passing retrieved context directly into the Responses API call."""
    system_prompt = (
        "You are Phi-4-reasoning, a helpful fitness AI.\n"
        "We have some context from the user's knowledge base: \n"
        f"{retrieved_doc}\n"
        "Please use this context to help your answer. If the context doesn't help, say so.\n"
    )

    response = openai_client.responses.create(
        model=phi4_deployment,
        instructions=system_prompt,
        input=user_question,
        temperature=0.3,
        max_output_tokens=300,
    )

    parsed = split_reasoning_and_answer(response.output_text)
    return parsed if show_thinking else parsed["answer"]

# Define a dummy doc snippet:
doc_snippet = (
    "Recommended to run 3 times per week and mix with cross-training.\n"
    "Include rest days or active recovery days for muscle repair."
)

user_q = "How often should I run weekly to prepare for a 5K?"
rag_answer = chat_with_phi4_rag(user_q, doc_snippet)
print("🗣️ User:", user_q)
print("🤖 Phi-4-reasoning (RAG):", rag_answer)

## 5. Wrap-Up & Best Practices

- **Step-by-Step Prompts:** Asking for explicit working can help on math or planning tasks — decide what to surface to end users. Use `show_thinking=True` during development/debugging, and keep it `False` (the default) for anything user-facing.
- **Reasoning traces:** Always separate `<think>...</think>` content from the final answer before displaying output to a user — never show raw reasoning traces in a production UI.
- **RAG:** For real retrieval-augmented answers, use a Foundry Agent with a retrieval tool (see `3-basic-rag.ipynb`) rather than manually pasting context, as shown in the stub above.
- **OpenTelemetry:** Optionally integrate `opentelemetry-sdk` and `azure-core-tracing-opentelemetry` for full observability.
- **Evaluate:** Use `azure-ai-evaluation` to measure your model's performance.
- **Cost & Performance:** Phi-4 delivers strong results from a small 14B model at low cost. It shines on math/coding/STEM — evaluate for your domain needs.

## 🎉 Congratulations!

You've seen how to:
- Use Phi-4 with `AIProjectClient` and its OpenAI client (`get_openai_client()`) via the **Responses API**.
- Create a chat flow with an optional step-by-step prompt, with reasoning properly separated from the final answer.
- Stub a RAG scenario, and see how it differs from the real agent-based approach in `3-basic-rag.ipynb`.